# LLM Benchmark ELT

The following notebook involves an ELT pipeline that ingests LLM performance benchmarks (intelligence scores, latency, pricing) from the *Artificial Analysis API*.

Data flows through a medallion architecture: **Bronze** (raw) → **Silver** (cleaned Delta Lake table partitioned by snapshot date) → **Gold** (ranked leaderboard + trends).

## 1. Importing & Logging

In [ ]:
# Install dependencies
%pip install -r ../requirements.txt -q

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.core.logging import configure_logging
from src.core.config import load_config
from src.core.spark_session import get_or_create_spark
from src.bronze.ingest import fetch_models, flatten_data, save_raw
from src.silver.transform import transform
from src.silver.write_delta import write_to_delta, verify_delta_table

configure_logging("INFO")

## 2. Load Configuration

In [15]:
config = load_config()
print(f"Snapshot date: {config['snapshot_date']}")
print(f"Bronze dir:    {config['bronze_dir']}")
print(f"Silver dir:    {config['silver_dir']}")

Snapshot date: 2026-06-09
Bronze dir:    /mnt/c/Users/AbdallahYamani/Documents/Personal/llm-benchmark-transformation/data/bronze
Silver dir:    /mnt/c/Users/AbdallahYamani/Documents/Personal/llm-benchmark-transformation/data/silver


## 3. Extract — Fetch from API

In [16]:
raw_data = fetch_models(config["api_url"], config["api_key"])
print(f"Models fetched: {len(raw_data['data'])}")

2026-06-09 13:50:20 | src.bronze.ingest | INFO | Models fetched: 534 items


Models fetched: 534


## 4. Bronze — Flatten and Save Parquet

In [17]:
# Flatten to DataFrame
df_bronze = flatten_data(raw_data, config["snapshot_date"])
print(f"Bronze DataFrame: {len(df_bronze)} rows, {len(df_bronze.columns)} columns")
df_bronze.head()

Bronze DataFrame: 534 rows, 25 columns


,model_id,model_slug,release_date,creator_slug,price_1m_input_tokens,price_1m_output_tokens,median_output_tokens_per_second,median_time_to_first_token_seconds,median_time_to_first_answer_token,snapshot_date,...,eval_hle,eval_livecodebench,eval_scicode,eval_math_500,eval_aime,eval_aime_25,eval_ifbench,eval_lcr,eval_terminalbench_hard,eval_tau2
0,36f73aaf-d38a-4b56-a2b3-d04d17186910,gpt-oss-20b,2025-08-05,openai,0.05,0.20,266.972,0.416,7.908,2026-06-09,...,0.098,0.777,0.344,NaN,NaN,0.893333,0.651020,0.306667,0.106061,0.602339
1,16149b9c-a1e9-4669-a5cb-ff3c00d78f89,gpt-oss-20b-low,2025-08-05,openai,0.06,0.20,272.042,0.460,7.812,2026-06-09,...,0.051,0.652,0.340,NaN,NaN,0.623333,0.578231,0.310000,0.045455,0.502924
2,f0083258-8646-45b8-8082-7aaf6c2ea82a,gpt-oss-120b,2025-08-05,openai,0.15,0.60,367.827,0.513,5.951,2026-06-09,...,0.185,0.878,0.389,NaN,NaN,0.934417,0.689796,0.506667,0.234848,0.657895
3,c99f3bde-7c08-4de8-bd5c-8ee9123ebffa,gpt-oss-120b-low,2025-08-05,openai,0.15,0.60,373.523,0.509,5.863,2026-06-09,...,0.052,0.707,0.360,NaN,NaN,0.666667,0.582993,0.436667,0.053030,0.450292
4,d4fc3f33-f2b0-4da1-88ee-f1f82bd4de31,gpt-5-4-nano,2026-03-17,openai,0.20,1.25,155.348,4.079,4.079,2026-06-09,...,0.265,NaN,0.469,NaN,NaN,NaN,0.759184,0.660000,0.424242,0.760234


In [18]:
# Save Parquet
parquet_path = save_raw(df_bronze, config["bronze_dir"], config["snapshot_date"])
print(f"Parquet saved: {parquet_path}")

2026-06-09 13:50:20 | src.bronze.ingest | INFO | Saved Parquet to /mnt/c/Users/AbdallahYamani/Documents/Personal/llm-benchmark-transformation/data/bronze/models_2026-06-09.parquet (534 rows)


Parquet saved: /mnt/c/Users/AbdallahYamani/Documents/Personal/llm-benchmark-transformation/data/bronze/models_2026-06-09.parquet


## 5. Spark Session

In [19]:
spark = get_or_create_spark()
print(f"Spark version: {spark.version}")

2026-06-09 13:50:20 | src.core.spark_session | INFO | SparkSession created successfully.


Spark version: 3.5.4


## 6. Silver Transform

In [20]:
df_silver = transform(parquet_path, spark)
df_silver.show(5, truncate=False)

2026-06-09 13:50:21 | src.silver.transform | INFO | Silver transform complete — 534 rows produced


+------------------------------------+----------------+------------+------------+---------------------+----------------------+-------------------------------+----------------------------------+---------------------------------+-------------+-------------------------------------------+-------------------------------------+-----------------------------------+-------------+---------+--------+------------------+------------+-------------+---------+-----------------+-----------------+-----------------+-----------------------+-----------------+
|model_id                            |model_name      |release_date|vendor      |price_1m_input_tokens|price_1m_output_tokens|median_output_tokens_per_second|median_time_to_first_token_seconds|median_time_to_first_answer_token|snapshot_date|eval_artificial_analysis_intelligence_index|eval_artificial_analysis_coding_index|eval_artificial_analysis_math_index|eval_mmlu_pro|eval_gpqa|eval_hle|eval_livecodebench|eval_scicode|eval_math_500|eval_aime|eval_ai

In [21]:
# Check schema
df_silver.printSchema()

root
 |-- model_id: string (nullable = true)
 |-- model_name: string (nullable = true)
 |-- release_date: date (nullable = true)
 |-- vendor: string (nullable = true)
 |-- price_1m_input_tokens: double (nullable = true)
 |-- price_1m_output_tokens: double (nullable = true)
 |-- median_output_tokens_per_second: double (nullable = true)
 |-- median_time_to_first_token_seconds: double (nullable = true)
 |-- median_time_to_first_answer_token: double (nullable = true)
 |-- snapshot_date: date (nullable = true)
 |-- eval_artificial_analysis_intelligence_index: double (nullable = true)
 |-- eval_artificial_analysis_coding_index: double (nullable = true)
 |-- eval_artificial_analysis_math_index: double (nullable = true)
 |-- eval_mmlu_pro: double (nullable = true)
 |-- eval_gpqa: double (nullable = true)
 |-- eval_hle: double (nullable = true)
 |-- eval_livecodebench: double (nullable = true)
 |-- eval_scicode: double (nullable = true)
 |-- eval_math_500: double (nullable = true)
 |-- eval_aim

## 7. Delta Write

In [22]:
write_to_delta(df_silver, config["silver_dir"])

2026-06-09 13:50:22 | src.silver.write_delta | INFO | Silver write complete — 534 rows persisted to /mnt/c/Users/AbdallahYamani/Documents/Personal/llm-benchmark-transformation/data/silver/models


## 8. Verify Delta Table

In [23]:
summary = verify_delta_table(config["silver_dir"], spark)
print(f"Rows: {summary['row_count']}")
print(f"Partitions: {summary['partition_count']}")
print(f"History entries: {len(summary['history'])}")

2026-06-09 13:50:26 | src.silver.write_delta | INFO | Silver verification passed — 534 rows, 1 partitions


Rows: 534
Partitions: 1
History entries: 3


## 9. Gold Build

In [ ]:
from src.gold.transform import compute_model_rankings, compute_model_trends
from src.gold.write_delta import write_gold_table

# Read Silver Delta for Gold
silver_path = str(config["silver_dir"] / "models")
df_all = spark.read.format("delta").load(silver_path)
df_snapshot = df_all.filter(df_all["snapshot_date"] == config["snapshot_date"])

# Build and write Gold tables
rankings = compute_model_rankings(df_snapshot, config["snapshot_date"])
write_gold_table(rankings, config["gold_dir"], "model_leaderboard")

trends = compute_model_trends(df_all)
write_gold_table(trends, config["gold_dir"], "model_trends")

## 10. Verify Gold

In [ ]:
gold_leaderboard = spark.read.format("delta").load(str(config["gold_dir"] / "model_leaderboard"))
gold_trends = spark.read.format("delta").load(str(config["gold_dir"] / "model_trends"))

print(f"Leaderboard: {gold_leaderboard.count()} rows")
print(f"Trends: {gold_trends.count()} rows")
gold_leaderboard.printSchema()

## 11. Analysis

In [ ]:
import matplotlib.pyplot as plt

# Top 10 models by composite score
top10 = gold_leaderboard.orderBy("rank").limit(10).toPandas()
print("Top 10 Models:")
print(top10[["rank", "model_name", "vendor", "composite_score", "cost_tier"]].to_string(index=False))

In [ ]:
df_plot = gold_leaderboard.filter("cost_tier IS NOT NULL").toPandas()

colors = {"budget": "green", "mid": "orange", "premium": "red"}
fig, ax = plt.subplots(figsize=(10, 6))
for tier, group in df_plot.groupby("cost_tier"):
    ax.scatter(group["avg_price_per_1m_tokens"], group["intelligence_norm"],
               label=tier, color=colors[tier], alpha=0.6)
               
ax.set_xlabel("Avg Price per 1M Tokens ($)")
ax.set_ylabel("Intelligence Score (normalized)")
ax.set_title("Intelligence vs Price by Cost Tier")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
spark.stop()